# Triton Integration for Hotel Search with Superlinked

This notebook demonstrates how to use Triton Inference Server with Superlinked for semantic hotel search.

## What you'll learn:
1. How to configure Triton gRPC connection
2. Use TextSimilaritySpace with Triton backend for embeddings
3. Build a hotel search system with semantic matching
4. Search hotels by description, amenities, and reviews

## Prerequisites:
- Triton server running with a text embedding model (e.g., qwen3-embedding-06B)
- `pip install superlinked[triton]` or `pip install tritonclient[grpc]`

## Use Case: Hotel Search
Search for hotels based on:
- **Description**: Hotel amenities and features
- **Reviews**: Guest experiences and feedback
- **Price & Rating**: Numeric filters combined with semantic search


## 1. Setup and Imports


In [5]:
# Import required libraries
import asyncio
from typing import List
from superlinked import framework as sl

# Superlinked imports
from sl import (
    TextSimilaritySpace, 
    TextModelHandler, 
    TritonEngineConfig,
    NumberSpace,
    Schema, 
    String,
    Float,
    Integer,
    Index,
    Query,
    InMemorySource,
    InMemoryExecutor,
    InMemoryVectorDatabase
)

print("✅ Imports successful!")


AttributeError: partially initialized module 'torchvision' has no attribute 'extension' (most likely due to a circular import)

## 2. Define Hotel Schema


In [ ]:
@Schema
class Hotel:
    """Hotel schema for semantic search."""
    id: str
    name: String
    description: String  # Hotel amenities, features, location details
    price: Float         # Price per night
    rating: Float        # Average rating (1.0 - 5.0)
    review: String       # Sample guest review/feedback
    location: String     # City/area location
    
print("✅ Hotel schema defined!")


## 3. Configure Triton Connection


In [ ]:
# Configure Triton gRPC connection
# Option 1: Use defaults from Settings (can be configured via env vars or config.yaml)
triton_config = TritonEngineConfig()

# Option 2: Override specific values
# triton_config = TritonEngineConfig(
#     grpc_url="localhost:6565",      # Your Triton server
#     model_name="qwen3-embedding-06B", # Your embedding model
#     timeout_seconds=30.0,            # Request timeout
#     max_retries=3                    # Retry attempts
# )

print("🔧 Triton Configuration:")
print(f"   - gRPC URL: {triton_config.triton_grpc_url}")
print(f"   - Model: {triton_config.triton_model_name}")
print(f"   - Version: {triton_config.triton_model_version}")
print(f"   - Timeout: {triton_config.triton_timeout_seconds}s")
print(f"   - Max Retries: {triton_config.triton_max_retries}")


## 4. Create Embedding Spaces with Triton Backend


In [ ]:
# Text similarity space for hotel descriptions using Triton
description_space = TextSimilaritySpace(
    text=Hotel.description,
    model="qwen3-embedding-06B",  # Should match your Triton model
    model_handler=TextModelHandler.TRITON,  # Use Triton backend
    embedding_engine_config=triton_config,
    cache_size=1000
)

# Text similarity space for reviews
review_space = TextSimilaritySpace(
    text=Hotel.review,
    model="qwen3-embedding-06B",
    model_handler=TextModelHandler.TRITON,
    embedding_engine_config=triton_config,
    cache_size=1000
)

# Number spaces for price and rating filtering
price_space = NumberSpace(
    number=Hotel.price,
    min_value=0,
    max_value=1000
)

rating_space = NumberSpace(
    number=Hotel.rating,
    min_value=1.0,
    max_value=5.0
)

print("✅ Embedding spaces created with Triton backend!")
print("   🏨 Description space: Semantic search on hotel features")
print("   💬 Review space: Search based on guest experiences")
print("   💰 Price space: Numerical filtering")
print("   ⭐ Rating space: Quality filtering")


## 5. Sample Hotel Data


In [ ]:
# Sample hotel data for demonstration
hotels = [
    Hotel(
        id="hotel_1",
        name="Luxury Beach Resort",
        description="Beachfront luxury resort with private beach access, infinity pool, spa services, fine dining restaurants, and water sports facilities. Perfect for romantic getaways.",
        price=450.0,
        rating=4.8,
        review="Amazing beachfront location with incredible sunset views. The spa was relaxing and the food was exceptional. Perfect for our honeymoon!",
        location="Malibu, California"
    ),
    Hotel(
        id="hotel_2",
        name="Downtown Business Hotel",
        description="Modern business hotel in city center with conference rooms, business center, fitness gym, and high-speed wifi. Walking distance to financial district.",
        price=180.0,
        rating=4.2,
        review="Great location for business travel. Fast wifi, comfortable rooms, and excellent conference facilities. Staff was very professional.",
        location="New York City, New York"
    ),
    Hotel(
        id="hotel_3",
        name="Mountain Lodge Retreat",
        description="Cozy mountain lodge with hiking trails, fireplace lounge, hot tub, mountain views, and outdoor adventure activities. Family-friendly with kids activities.",
        price=220.0,
        rating=4.5,
        review="Perfect family vacation spot! Kids loved the outdoor activities and we enjoyed the peaceful mountain setting. Hot tub was a nice touch after hiking.",
        location="Aspen, Colorado"
    ),
    Hotel(
        id="hotel_4",
        name="Budget City Inn",
        description="Affordable city hotel with basic amenities, free wifi, continental breakfast, and convenient public transport access. Clean and comfortable rooms.",
        price=85.0,
        rating=3.8,
        review="Great value for money. Clean rooms, friendly staff, and good location near metro station. Perfect for budget travelers.",
        location="Chicago, Illinois"
    ),
    Hotel(
        id="hotel_5",
        name="Boutique Art Hotel",
        description="Stylish boutique hotel featuring local art, rooftop bar, gourmet restaurant, unique themed rooms, and personalized service in historic district.",
        price=320.0,
        rating=4.6,
        review="Absolutely loved the artistic atmosphere and unique room design. Rooftop bar has amazing cocktails and city views. Very Instagram-worthy!",
        location="Santa Fe, New Mexico"
    ),
    Hotel(
        id="hotel_6",
        name="Spa Wellness Resort",
        description="Health-focused resort with full-service spa, yoga classes, meditation gardens, organic restaurant, wellness programs, and holistic treatments.",
        price=380.0,
        rating=4.7,
        review="The perfect place to recharge and relax. Yoga classes were amazing and the spa treatments were world-class. Felt completely rejuvenated!",
        location="Sedona, Arizona"
    )
]

print(f"📊 Created {len(hotels)} sample hotels for search:")
for hotel in hotels:
    print(f"   🏨 {hotel.name} - ${hotel.price}/night - ⭐{hotel.rating} - {hotel.location}")


## 6. Setup Search System and Run Example Searches

This cell demonstrates the complete setup and several search examples using Triton-powered embeddings.


In [ ]:
# Create index with all spaces
hotel_index = Index([description_space, review_space, price_space, rating_space])

# Create data source and executor
hotel_source = InMemorySource(Hotel)
executor = InMemoryExecutor(
    sources=[hotel_source],
    indices=[hotel_index],
    vector_database=InMemoryVectorDatabase()
)

# Run the application
app = executor.run()

# Add hotels to the system
app.source.put(hotels)
print(f"✅ Loaded {len(hotels)} hotels with Triton-powered embeddings")

# Example 1: Search for romantic beach getaway
print("\n🌅 Search: Romantic Beach Getaway")
print("=" * 50)
romantic_query = (
    Query(description_space)
    .similar("romantic beachfront resort with spa and sunset views", weight=1.0)
    .similar(review_space, "honeymoon romantic sunset amazing views", weight=0.7)
)
results = app.query(romantic_query, limit=2)
for result in results:
    hotel = result.entity
    print(f"🏨 {hotel.name} (${hotel.price}/night, ⭐{hotel.rating})")
    print(f"📍 {hotel.location}")
    print(f"📝 {hotel.description[:100]}...")
    print(f"💬 {hotel.review[:80]}...")
    print("-" * 30)

# Example 2: Budget travel with good value
print("\n💰 Search: Budget Travel Under $150 with Good Reviews")
print("=" * 50)
budget_query = (
    Query(description_space)
    .similar("budget affordable clean comfortable", weight=0.8)
    .similar(review_space, "great value for money", weight=1.0)
    .filter(price_space < 150)
    .filter(rating_space > 3.5)
)
results = app.query(budget_query, limit=2)
for result in results:
    hotel = result.entity
    print(f"🏨 {hotel.name} (${hotel.price}/night, ⭐{hotel.rating})")
    print(f"📍 {hotel.location}")
    print(f"📝 {hotel.description[:100]}...")
    print(f"💬 {hotel.review[:80]}...")
    print("-" * 30)

print("\n🎉 Triton integration working successfully!")
print("🔍 Try modifying the search queries above to explore different results")
